### Testing GraphBuilder

In [2]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
import torch

dset = datasets.Planetoid(root=f'./../res/datasets/cora', name='cora')


data = dset[0]

train_mask, val_mask, test_mask = TrainValTestSplitter.split('cora', dset)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=dset.num_classes, 
            num_features=dset.num_features, 
            hidden_size=16)

model.set_classification_task(task='node')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)
model.train()
epoch_count = 200
for epoch in range(epoch_count):
    # Clear the gradients
    optimizer.zero_grad()
    # Forward Pass
    out = model(data.x, data.edge_index)
    # Find the Loss
    train_loss = criterion(out[train_mask], data.y[train_mask])    
    # Calculate gradients
    train_loss.backward()
    # Update Weights
    optimizer.step()
    print(f"""Epoch: [{epoch}/{epoch_count}]
    Training loss: {train_loss.item():.4f}""")

    ### VALIDATION
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        val_loss = criterion(out[val_mask], data.y[val_mask])
        _, pred = torch.max(out.data, 1)
        correct = (pred[val_mask] == data.y[val_mask]).float().sum().item()
        acc = correct / val_mask.shape[0]
    
    if (epoch + 1) % 10 == 0:
        print(f"""Epoch: [{epoch+1}/{epoch_count}]
          Training loss: {train_loss.item():.4f},\t Validation loss: {val_loss.item():.4f}
          Accuracy: {acc}""")

Epoch: [0/200]
    Training loss: 1.9597
Epoch: [1/200]
    Training loss: 1.9406
Epoch: [2/200]
    Training loss: 1.9213
Epoch: [3/200]
    Training loss: 1.8984
Epoch: [4/200]
    Training loss: 1.8723
Epoch: [5/200]
    Training loss: 1.8433
Epoch: [6/200]
    Training loss: 1.8122
Epoch: [7/200]
    Training loss: 1.7791
Epoch: [8/200]
    Training loss: 1.7439
Epoch: [9/200]
    Training loss: 1.7067
Epoch: [10/200]
          Training loss: 1.7067,	 Validation loss: 1.6638
          Accuracy: 0.3929889298892989
Epoch: [10/200]
    Training loss: 1.6678
Epoch: [11/200]
    Training loss: 1.6273
Epoch: [12/200]
    Training loss: 1.5852
Epoch: [13/200]
    Training loss: 1.5416
Epoch: [14/200]
    Training loss: 1.4965
Epoch: [15/200]
    Training loss: 1.4500
Epoch: [16/200]
    Training loss: 1.4021
Epoch: [17/200]
    Training loss: 1.3531
Epoch: [18/200]
    Training loss: 1.3031
Epoch: [19/200]
    Training loss: 1.2522
Epoch: [20/200]
          Training loss: 1.2522,	 Validat

In [26]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from rewiring.model.GCN import GCN
import torch
import numpy as np

data = datasets.TUDataset(root=f'./../res/datasets/proteins', name='PROTEINS')

train_size = int(0.8 * len(data))
val_size = len(data) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(data, [train_size, val_size])

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(num_classes=data.num_classes, 
            num_features=data.num_features, 
            hidden_size=16)
model.set_classification_task(task='graph')
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

model.to(device=device)

acc_list = []
train_losses = []
val_losses = []

epoch_count = 200
for epoch in range(1, epoch_count + 1):

    epoch_train_loss = []
    epoch_val_loss = []

    ### TRAINING
    model.train()
    for batch in train_dloader:
        # Clear the gradients
        optimizer.zero_grad()
        # Forward Pass
        out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
        # Find the Loss
        train_loss = criterion(out, batch.y)
        epoch_train_loss.append(train_loss.item())    
        # Calculate gradients
        train_loss.backward()
        # Update Weights
        optimizer.step()

    ### VALIDATION
    model.eval()
    correct = 0
    for batch in val_dloader:
        with torch.no_grad():
            out = model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
            val_loss = criterion(out, batch.y)
            epoch_val_loss.append(val_loss.item())
            _, pred = torch.max(out.data, 1)
            correct += (pred == batch.y).float().sum().item()

    acc = correct / val_size
    acc_list.append(acc)

    epoch_train_loss = np.array(epoch_train_loss).mean().item()
    epoch_val_loss = np.array(epoch_val_loss).mean().item()
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
        
    if epoch % 10 == 0:
            print(f"""Epoch: [{epoch}/{epoch_count}] \
        Training loss: {epoch_train_loss:.4f} \
        Validation loss: {epoch_val_loss:.4f} \
        Accuracy: {acc:.4f}""")

Epoch: [10/200]         Training loss: 0.5907         Validation loss: 0.5546         Accuracy: 0.7085
Epoch: [20/200]         Training loss: 0.5537         Validation loss: 0.5112         Accuracy: 0.7489
Epoch: [30/200]         Training loss: 0.5450         Validation loss: 0.5129         Accuracy: 0.7534
Epoch: [40/200]         Training loss: 0.5456         Validation loss: 0.5039         Accuracy: 0.7534
Epoch: [50/200]         Training loss: 0.5402         Validation loss: 0.5038         Accuracy: 0.7578
Epoch: [60/200]         Training loss: 0.5428         Validation loss: 0.5029         Accuracy: 0.7623
Epoch: [70/200]         Training loss: 0.5418         Validation loss: 0.5188         Accuracy: 0.7444
Epoch: [80/200]         Training loss: 0.5344         Validation loss: 0.5022         Accuracy: 0.7623
Epoch: [90/200]         Training loss: 0.5331         Validation loss: 0.5063         Accuracy: 0.7534
Epoch: [100/200]         Training loss: 0.5355         Validation loss: 0